# CliniqAI — brain_mri Specialist v2 Retrain

Retrains the `brain_mri` specialist from scratch using the labeled data from
the separate Brain_MRI project (`E:\programming\Brain_MRI\data\raw`),
expanding beyond CliniqAI's current 4-class taxonomy
(`glioma, meningioma, notumor, pituitary`).

**Does NOT reuse that project's checkpoint** — different key prefixes
(`backbone.*`), different wide-head hidden dim (256 vs CliniqAI's 512), and
no `class_names`/`classes` key. Retraining sidesteps all three since CliniqAI
controls the checkpoint format this time.

Class names are derived directly from subfolder names via `ImageFolder`-style
scanning — never manually re-typed — to avoid the label-order bugs already
hit twice on this project (`kidney_ct`, `pneumonia`).

Run Step 1 first and read its output before doing anything else — it answers
the 3 open questions (dataset layout, notumor equivalent, class balance).

In [ ]:
# Step 1 — inspect the dataset structure BEFORE assuming anything.
# Answers all 3 open questions: is it ImageFolder-style, is there a
# no-tumor/normal class already, and how imbalanced are the classes.

from pathlib import Path

DATA_ROOT = Path(r"E:\programming\Brain_MRI\data\raw")

assert DATA_ROOT.exists(), f"Path not found: {DATA_ROOT} — fix DATA_ROOT before continuing"

subdirs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f"Found {len(subdirs)} subfolders under {DATA_ROOT}:\n")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
counts = {}
for d in subdirs:
    n = sum(1 for f in d.rglob("*") if f.suffix.lower() in IMG_EXT)
    counts[d.name] = n
    print(f"  {d.name:35s} {n:6d} images")

total = sum(counts.values())
print(f"\nTotal: {total} images across {len(counts)} classes")

if counts:
    mx, mn = max(counts.values()), min(counts.values())
    print(f"Class imbalance ratio (max/min): {mx/mn:.1f}x" if mn else "min count is 0 — check that folder")

# Manually check this printed list for anything resembling "notumor" / "normal" /
# "no_tumor" / "healthy". If nothing like that appears, set MERGE_NOTUMOR = True
# below to pull it in from CliniqAI's own data/raw/brain_tumor/notumor folder.
print("\n--> Check the class list above for a no-tumor/normal class before continuing.")

## Step 1 result — action needed

Read the printed folder names and counts above, then set the two flags in
the next cell by hand:

- `MERGE_NOTUMOR` — `True` if no no-tumor/normal-equivalent class was in the
  list, `False` if one already exists (e.g. `notumor`, `no_tumor`, `normal`).
- If imbalance ratio (max/min) printed above is large (rule of thumb: >10x),
  we'll use a `WeightedRandomSampler` in Step 6 rather than plain shuffling.

In [ ]:
# Step 2 — flags based on Step 1's output. EDIT THESE before running further.

MERGE_NOTUMOR = True  # set False if Step 1 already showed a notumor/normal class
NOTUMOR_SOURCE_DIR = r"E:\programming\CliniqAI\data\raw\brain_tumor\notumor"
NOTUMOR_CLASS_NAME = "notumor"

USE_WEIGHTED_SAMPLER = True  # leave True unless Step 1 showed near-balanced classes

In [ ]:
# Step 3 — imports, reproducibility, device + CPU thread setup.
# "Maximum CPU power within VRAM limits" = maximize DataLoader worker
# parallelism (CPU-side decode/augment) while keeping GPU batch size/AMP
# settings within the proven-safe 6GB envelope from the other specialists.

import os
import sys
import random
import copy
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sys.path.append(str(Path(r"E:\programming\CliniqAI").resolve()))
from scripts.brain_mri_v2_dataset import BrainMRIDataset, build_samples, validate_images

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # established gotcha fix, keep it

# i5-13450HX: 10 cores / 16 threads. Leave 2 threads for the main process
# and OS so the machine stays responsive; give the rest to DataLoader workers.
CPU_COUNT = os.cpu_count() or 8
NUM_WORKERS = max(4, CPU_COUNT - 2)
torch.set_num_threads(max(1, CPU_COUNT - NUM_WORKERS))  # main-process compute threads

print(f"Device: {DEVICE}")
print(f"CPU count: {CPU_COUNT} -> NUM_WORKERS={NUM_WORKERS}, main-thread compute threads={max(1, CPU_COUNT - NUM_WORKERS)}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Step 4 — build the sample list + class names directly from disk.

samples, class_names, class_to_idx = build_samples(DATA_ROOT)
print(f"{len(samples)} samples across {len(class_names)} classes (from folder names):")
for name, idx in class_to_idx.items():
    print(f"  {idx}: {name}")

if MERGE_NOTUMOR:
    notumor_dir = Path(NOTUMOR_SOURCE_DIR)
    assert notumor_dir.exists(), f"NOTUMOR_SOURCE_DIR not found: {notumor_dir}"

    new_idx = len(class_names)
    class_names = class_names + [NOTUMOR_CLASS_NAME]
    class_to_idx[NOTUMOR_CLASS_NAME] = new_idx

    IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    notumor_samples = [
        (str(f), new_idx) for f in notumor_dir.rglob("*") if f.suffix.lower() in IMG_EXT
    ]
    samples += notumor_samples
    print(f"\nMerged {len(notumor_samples)} notumor images from {notumor_dir} as class {new_idx}")

print(f"\nFinal: {len(samples)} samples, {len(class_names)} classes: {class_names}")

In [ ]:
# Step 5 — pre-validation pass: drop corrupted/unreadable files up front
# rather than hitting them mid-epoch (established gotcha).

samples = validate_images(samples)
print(f"\n{len(samples)} samples remain after validation.")

In [ ]:
# Step 6 — stratified train/val/test split (80/10/10), then check
# imbalance and build a WeightedRandomSampler for training if flagged.

labels = [label for _, label in samples]

train_samples, temp_samples, train_labels, temp_labels = train_test_split(
    samples, labels, test_size=0.20, stratify=labels, random_state=SEED
)
val_samples, test_samples, val_labels, test_labels = train_test_split(
    temp_samples, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)

print(f"Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}")

class_counts = np.bincount(train_labels, minlength=len(class_names))
print("\nPer-class training counts:")
for name, c in zip(class_names, class_counts):
    print(f"  {name:25s} {c:6d}")

train_sampler = None
if USE_WEIGHTED_SAMPLER:
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = [class_weights[label] for label in train_labels]
    train_sampler = WeightedRandomSampler(
        sample_weights, num_samples=len(train_samples), replacement=True
    )
    print("\nUsing WeightedRandomSampler for training (rare classes upsampled).")

# Loss-side class weights too (belt-and-suspenders with the sampler is fine —
# drop one if double-correction looks like it's hurting majority-class recall).
loss_class_weights = torch.tensor(
    1.0 / np.maximum(class_counts, 1), dtype=torch.float32
)
loss_class_weights = loss_class_weights / loss_class_weights.sum() * len(class_names)
loss_class_weights = loss_class_weights.to(DEVICE)

In [ ]:
# Step 7 — transforms. 224x224, ImageNet normalization (established recipe).
# Mild augmentation on train only — MRI images shouldn't get heavy color jitter
# (real scans don't vary in hue/saturation the way natural photos do).

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
# Step 8 — Datasets + DataLoaders.
# num_workers=NUM_WORKERS maxes out CPU-side decode/augment throughput so the
# GPU isn't starved; pin_memory + persistent_workers + prefetch_factor=4 are
# the established combo for this. Batch size 64 is the proven-safe ceiling on
# the RTX 4050's 6GB with AMP — see Step 9/10.

BATCH_SIZE = 64

train_ds = BrainMRIDataset(train_samples, transform=train_transform)
val_ds = BrainMRIDataset(val_samples, transform=eval_transform)
test_ds = BrainMRIDataset(test_samples, transform=eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    sampler=train_sampler, shuffle=(train_sampler is None),
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=4,
)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

In [ ]:
# Step 9 — model: EfficientNet-B0, wide head (hidden=512), matching the
# other 3 wide-head specialists so model_loader.py's _build_matching_head()
# picks it up without changes.

def build_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )
    return model

model = build_model(len(class_names)).to(DEVICE)
model = model.to(memory_format=torch.channels_last)  # free speed win on Ada (RTX 4050)

print(model.classifier)

In [ ]:
# Step 10 — AMP, optimizer, scheduler, early stopping setup.
# AMP roughly halves activation memory, which is what makes batch 64 safe on
# 6GB VRAM (batch 64 without AMP previously overflowed into slow shared
# system memory on this same GPU for other specialists).

from torch.cuda.amp import autocast, GradScaler

criterion = nn.CrossEntropyLoss(weight=loss_class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)
scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

EARLY_STOP_PATIENCE = 8
MAX_EPOCHS = 60

In [ ]:
# Step 11 — training loop with early stopping + best-checkpoint tracking.

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})
    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no val improvement for {EARLY_STOP_PATIENCE} epochs)")
            break

model.load_state_dict(best_state)
print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# Step 12 — final evaluation on the held-out test set: overall accuracy +
# per-class breakdown (not just one aggregate number — established convention
# after the gatekeeper breast-histology miss).

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with autocast(enabled=(DEVICE.type == "cuda")):
            outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
# Step 13 — save the checkpoint in CliniqAI's expected format:
# model_state_dict + class_names + classes (redundant, matches pneumonia's
# working checkpoint), wide head hidden=512 so _build_matching_head() picks
# it up with no loader changes needed.

OUT_PATH = Path(r"E:\programming\CliniqAI\models\specialists\brain_tumor_specialist_v2.pt")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "classes": class_names,
    "num_classes": len(class_names),
    "architecture": "efficientnet_b0_wide512",
}
torch.save(checkpoint, OUT_PATH)
print(f"Saved to {OUT_PATH}")

with open(OUT_PATH.with_suffix(".history.json"), "w") as f:
    json.dump(history, f, indent=2)
print("Training history saved alongside checkpoint.")

print("""
Next steps (not done by this notebook):
1. Copy checkpoint into backend/models/specialists/ (manual sync, no build step yet).
2. Point config.py's brain_mri specialist entry at brain_tumor_specialist_v2.pt.
3. Update KNOWN_CLASS_NAMES / confirm model_loader.py picks up the new class list
   via the classes key (same path that fixed pneumonia).
4. Re-run the acoustic neuroma / out-of-distribution sanity test that originally
   flagged the 4-class limitation, plus a fresh manual_sanity_check pass.
5. Upload the new checkpoint to the shared Google Drive so Roneet and Khushboo can pull it.
""")